# LACCI figures notebook

This notebook reads the exported CSV files from `experiments_lacci_rerun.ipynb` and produces paper-ready figures using **seaborn**.

Update `RESULTS_DIR` below if needed, then run top-to-bottom.


In [ ]:
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["savefig.bbox"] = "tight"

RESULTS_DIR = Path("minimal_results_cc18_lacci")
FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR, FIG_DIR


## Helpers

In [ ]:
def find_first(pattern: str):
    matches = sorted(RESULTS_DIR.rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"No files matched: {pattern}")
    return matches[0]

def find_all(pattern: str):
    matches = sorted(RESULTS_DIR.rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"No files matched: {pattern}")
    return matches

def normalize_representation_name(name: str) -> str:
    if not isinstance(name, str):
        return str(name)
    s = name.lower()
    if "tfidf" in s:
        return "TF-IDF"
    if "minilm" in s:
        return "MiniLM"
    if "meta" in s and "only" in s:
        return "Meta-only"
    if s in {"none", "metafeatures", "meta_features"}:
        return "Meta-only"
    return name

def normalize_model_name(name: str) -> str:
    if not isinstance(name, str):
        return str(name)
    s = name.lower()
    mapping = {
        "hist_gbrt": "HistGBRT",
        "random_forest": "Random Forest",
        "extra_trees": "Extra Trees",
        "ridge": "Ridge",
    }
    return mapping.get(s, name)

def infer_run_labels(path: Path):
    parts = [p.lower() for p in path.parts]
    rep = "Unknown"
    if any("tfidf" in p for p in parts):
        rep = "TF-IDF"
    elif any("minilm" in p for p in parts):
        rep = "MiniLM"
    elif any(("meta" in p and "only" in p) or p == "none" for p in parts):
        rep = "Meta-only"
    model = "Unknown"
    for key, label in [("hist_gbrt", "HistGBRT"), ("random_forest", "Random Forest"),
                       ("extra_trees", "Extra Trees"), ("ridge", "Ridge")]:
        if key in "_".join(parts):
            model = label
    return rep, model


## Load summary tables

In [ ]:
anchor_path = find_first("lacci_predictive_anchor.csv")
shap_summary_path = find_first("lacci_shap_summary_metrics.csv")
local_summary_path = find_first("lacci_local_shap_metrics_summary.csv")

anchor_df = pd.read_csv(anchor_path)
shap_summary_df = pd.read_csv(shap_summary_path)
local_summary_df = pd.read_csv(local_summary_path)

display(anchor_df.head())
display(shap_summary_df.head())
display(local_summary_df.head())


In [ ]:
for df in [anchor_df, shap_summary_df, local_summary_df]:
    for col in df.columns:
        if "representation" in col.lower():
            df[col] = df[col].map(normalize_representation_name)
        if col.lower() == "model":
            df[col] = df[col].map(normalize_model_name)

anchor_df.columns.tolist(), shap_summary_df.columns.tolist(), local_summary_df.columns.tolist()


## Figure 1 — Predictive anchor

This produces a compact comparison of predictive performance for the selected configurations.


In [ ]:
metric_col = "r2_mean" if "r2_mean" in anchor_df.columns else ("r2" if "r2" in anchor_df.columns else None)
err_col = "r2_std" if "r2_std" in anchor_df.columns else None
rep_col = next(c for c in anchor_df.columns if "representation" in c.lower())
model_col = "model"

if metric_col is None:
    raise ValueError(f"Could not find R² column in: {anchor_df.columns.tolist()}")

plot_df = anchor_df.copy().sort_values([model_col, rep_col])

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=plot_df, x=rep_col, y=metric_col, hue=model_col, ax=ax)

if err_col is not None:
    # add manual error bars in the plotted order
    patches = ax.patches
    errs = []
    for _, row in plot_df.iterrows():
        errs.append(row[err_col])
    for patch, err in zip(patches, errs):
        x = patch.get_x() + patch.get_width()/2
        y = patch.get_height()
        ax.errorbar(x, y, yerr=err, color='black', capsize=4, linewidth=1)

ax.set_title("Predictive anchor for selected configurations")
ax.set_xlabel("")
ax.set_ylabel("Mean $R^2$")
ax.legend(title="")
plt.xticks(rotation=0)
plt.tight_layout()
out = FIG_DIR / "figure1_predictive_anchor.png"
plt.savefig(out, dpi=300)
plt.show()
print(out)


## Figure 2 — Global SHAP comparison

Reads per-run global SHAP CSV files and plots the top features for each representation/model.


In [ ]:
global_paths = find_all("global_shap_*_*.csv")
global_frames = []
for p in global_paths:
    df = pd.read_csv(p)
    rep, model = infer_run_labels(p)
    df["representation"] = rep
    df["model"] = model
    df["source_file"] = p.name
    global_frames.append(df)

global_df = pd.concat(global_frames, ignore_index=True)
display(global_df.head())
print(sorted(global_df["representation"].unique()), sorted(global_df["model"].unique()))


In [ ]:
top_k = 15
plot_global = (
    global_df.sort_values("mean_abs_shap", ascending=False)
             .groupby(["representation", "model"], as_index=False, group_keys=False)
             .head(top_k)
             .copy()
)

g = sns.catplot(
    data=plot_global,
    kind="bar",
    x="mean_abs_shap",
    y="feature",
    col="representation",
    row="model",
    sharex=False,
    height=5,
    aspect=1.4,
    palette="deep",
)
g.set_axis_labels("Mean |SHAP|", "")
g.set_titles(row_template="{row_name}", col_template="{col_name}")
for ax in g.axes.flat:
    ax.tick_params(axis="y", labelsize=10)
g.fig.subplots_adjust(top=0.93)
g.fig.suptitle("Top global SHAP features by representation and model")
out = FIG_DIR / "figure2_global_shap_comparison.png"
g.savefig(out, dpi=300)
plt.show()
print(out)


## Figure 3 — Local explanation quality

Uses the aggregated local SHAP metrics to compare readability and sparsity.


In [ ]:
df = local_summary_df.copy()
# guess useful columns
rep_col = next(c for c in df.columns if "representation" in c.lower())
model_col = "model"
read_col = next((c for c in df.columns if "readability" in c.lower()), None)
sparse_col = next((c for c in df.columns if "sparsity" in c.lower()), None)

if read_col is None or sparse_col is None:
    raise ValueError(f"Need readability and sparsity columns in {df.columns.tolist()}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=df, x=rep_col, y=read_col, hue=model_col, ax=axes[0], palette="deep")
axes[0].set_title("Explanation readability")
axes[0].set_xlabel("")
axes[0].set_ylabel("Mean readability")

sns.barplot(data=df, x=rep_col, y=sparse_col, hue=model_col, ax=axes[1], palette="deep")
axes[1].set_title("Explanation sparsity")
axes[1].set_xlabel("")
axes[1].set_ylabel("Features needed for 80% SHAP mass")

handles, labels = axes[1].get_legend_handles_labels()
axes[0].legend_.remove()
axes[1].legend(handles, labels, title="")
plt.tight_layout()
out = FIG_DIR / "figure3_local_quality.png"
plt.savefig(out, dpi=300)
plt.show()
print(out)


## Figure 4 — Global top-k stability across folds

Reads `global_shap_stability_*.csv` files and compares Jaccard overlap across representations/models.


In [ ]:
stability_paths = find_all("global_shap_stability_*_*.csv")
stability_frames = []
for p in stability_paths:
    df = pd.read_csv(p)
    rep, model = infer_run_labels(p)
    df["representation"] = rep
    df["model"] = model
    df["source_file"] = p.name
    stability_frames.append(df)
stability_df = pd.concat(stability_frames, ignore_index=True)
display(stability_df.head())


In [ ]:
# choose a likely stability column
stab_col = next((c for c in stability_df.columns if "jaccard" in c.lower() or "stability" in c.lower()), None)
if stab_col is None:
    raise ValueError(f"Could not infer stability column from {stability_df.columns.tolist()}")

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=stability_df, x="representation", y=stab_col, hue="model", ax=ax, palette="deep")
ax.set_title("Global SHAP top-k stability across folds")
ax.set_xlabel("")
ax.set_ylabel("Mean Jaccard overlap")
ax.legend(title="")
plt.tight_layout()
out = FIG_DIR / "figure4_global_stability.png"
plt.savefig(out, dpi=300)
plt.show()
print(out)


## Figure 5 — Interaction summary (optional)

This section runs only if interaction CSV files exist.


In [ ]:
interaction_paths = sorted(RESULTS_DIR.rglob("interaction_metrics_*_*.csv"))
if interaction_paths:
    interaction_frames = []
    for p in interaction_paths:
        df = pd.read_csv(p)
        rep, model = infer_run_labels(p)
        df["representation"] = rep
        df["model"] = model
        df["source_file"] = p.name
        interaction_frames.append(df)
    interaction_df = pd.concat(interaction_frames, ignore_index=True)
    display(interaction_df.head())

    ratio_col = next((c for c in interaction_df.columns if "ratio" in c.lower()), None)
    if ratio_col is None:
        raise ValueError(f"Could not infer interaction ratio column from {interaction_df.columns.tolist()}")

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=interaction_df, x="representation", y=ratio_col, hue="model", ax=ax, palette="deep")
    ax.set_title("Interaction strength summary")
    ax.set_xlabel("")
    ax.set_ylabel("Off-diagonal interaction ratio")
    ax.legend(title="")
    plt.tight_layout()
    out = FIG_DIR / "figure5_interactions.png"
    plt.savefig(out, dpi=300)
    plt.show()
    print(out)
else:
    print("No interaction metric files found. Skipping Figure 5.")


## Save compact paper tables

These CSVs are convenient for direct insertion into the manuscript or for additional polishing elsewhere.


In [ ]:
anchor_out = FIG_DIR / "table_predictive_anchor.csv"
local_out = FIG_DIR / "table_local_quality.csv"
stability_out = FIG_DIR / "table_global_stability.csv"

anchor_df.to_csv(anchor_out, index=False)
local_summary_df.to_csv(local_out, index=False)
stability_df.to_csv(stability_out, index=False)

print(anchor_out)
print(local_out)
print(stability_out)
